# Validating DC OPF Model, April 14, 2025

In [1]:
import pandas as pd
import pandapower as pp
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import copy
import copy
import pulp
import time
import pulp
import csv
import random

#MY CODE BASE
import GridFlowSmooth as flow 

datelabel = str(datetime.now())[2:10]

# PandaPower Solver as Reference
"joint development of the research group of the Department for Sustainable Electrical Energy Systems (e2n), University of Kassel and the Department for Distribution System Operation at the Fraunhofer Institute for Energy Economics and Energy System Technology (IEE), Kassel." 
https://pandapower.readthedocs.io/en/latest/


In [162]:
def panda_DCOPF(gen_capacity = {1:150,2:0,3:0,4:0}, line_capacity = 100, 
                 susceptances = {(1,2):1500, (2,3):1500, (3,4):1500, (1,3):1500, (1,4):1500}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4], obj = 'cost', cut = False,
                 demands = {1: 30, 2: 20, 3: 20, 4: 20}, ac_lines = [(1, 2), (2,3), (3,4)], 
                 dc_lines = [], generation_cost = {1: 120}, printout = True):
    net = pp.create_empty_network()
    ## # BUSES # ##
    pd_buses = {}             
    for i in buses:
        pd_buses[i] = pp.create_bus(net, vn_kv=1, name=f"Bus {i}")
    ## # LINES # ##
    for fb, tb in ac_lines:
        pp.create_line_from_parameters(net, from_bus=pd_buses[fb], to_bus=pd_buses[tb], length_km=1,
                                       r_ohm_per_km=0, x_ohm_per_km=1/susceptances[(fb,tb)], c_nf_per_km=0,
                                       max_i_ka=1000, name=f"Line {fb}-{tb}")
    ## # LOADS # ##
    for i in demands:
        pp.create_load(net, bus=pd_buses[i], p_mw=demands[i], q_mvar=0, controllable=False, name=f"Load Bus {i}")
    pd_gens = {}
    ## # GENERATORS # ##
    slack_gen = np.min(list(generation_cost.keys()))
    for i in gen_capacity:
        if gen_capacity[i]>0:
            pd_gens[i] = pp.create_gen(net, bus=pd_buses[i], p_mw=0, vm_pu=1.02, min_p_mw=0, max_p_mw=gen_capacity[i],
                         controllable=True, name=f"Gen Bus {i}", slack = i==slack_gen ,
                                       min_q_mvar = -10 , max_q_mvar=10)
    for i in pd_gens:
        pp.create_poly_cost(net, element=pd_gens[i], et="gen", cp1_eur_per_mw=generation_cost[i])
    # Run DC OPF
    out =pp.rundcopp(net, delta=1e-10, calculate_voltage_angles=True, trafo_model="pi")
    if printout:
        print("\n--- Generator Results ---")
        print(net.res_gen['p_mw'])
        
        print("\n--- Line Flow ---")
        print(net.res_line[["p_from_mw"]])
        
        print("\n--- Bus Voltage Angles (rad) ---")
        print(net.res_bus["va_degree"]*np.pi/180)
    return net

## Here is our model, running on python pulp which will soon use gurobi

In [7]:
out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:150,2:0,3:0,4:0}, line_capacity = 100, 
                 susceptances = {(1,2):1500, (2,3):1500, (3,4):1500, (1,3):1500, (1,4):1500}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4], obj = 'cost', cut = False,
                 demands = {1: 30, 2: 20, 3: 20, 4: 20}, ac_lines = [(1, 2), (2,3), (3,4)], 
                 poss_ac_lines = [(1, 3), (1,4)], dc_lines = [], poss_dc_lines = [(3,1)], 
                 generation_cost = {1: 120}, printout = True,
                 max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False)

Status: Optimal
Objective Value: 10800.0

Generation at Each Bus:
Bus 1: 90.0 MW

AC Line Flows:
Line 1-2: 60.0 MW
Line 2-3: 40.0 MW
Line 3-4: 20.0 MW
Line 1-3: 0.0 MW
Line 1-4: 0.0 MW

Voltage phases:
Bus 1: 0.0 rads
Bus 2: -0.039999999999999994 rads
Bus 3: -0.06666666666666667 rads
Bus 4: -0.08 rads


## Our results above resemble pandapower results below

In [10]:
net = panda_DCOPF()


--- Generator Results ---
0    90.0
Name: p_mw, dtype: float64

--- Line Flow ---
   p_from_mw
0       60.0
1       40.0
2       20.0

--- Bus Voltage Angles (rad) ---
0    0.000000
1   -0.040000
2   -0.066667
3   -0.080000
Name: va_degree, dtype: float64


# Now let's run a general test

In [116]:
bla = [1,2,3]
bla.remove(2)
print(bla)
bla.append(2)
print(bla)

[1, 3]
[1, 3, 2]


In [206]:
def get_partners(pairs, node):
    left = [pair[0] for pair in pairs if node==pair[1]]
    right = [pair[1] for pair in pairs if node==pair[0]]
    return left+right

def new_pairs(pairs, buses, n=3):
    out = []
    complete = [] 
    while len(out) < n and len(pairs + out) < (len(buses)-1)*len(buses)/2:
        node1 = random.choice(list(set(buses) - set(complete)))
        partners = get_partners(pairs+out, node1)
        if len(partners)<len(buses)-1:
            node2 = random.choice(list(set(buses) - set(partners + [node1])))
            out.append((node1, node2))
        else:
            complete.append(node1)
    return out
            

In [176]:
get_partners(ac_lines,4)

[3, 1, 0]

In [178]:
ac_lines

[(2, 1), (3, 4), (1, 3), (4, 1), (4, 0), (2, 0), (0, 3), (2, 3), (1, 0)]

In [202]:
new_pairs([], buses, 5)

[(2, 3), (1, 3), (2, 0), (7, 1), (3, 6)]

In [156]:
linees.pop()

(4, 2)

In [158]:
linees

[(2, 1), (3, 4), (1, 3), (4, 1), (4, 0), (2, 0), (0, 3), (2, 3), (1, 0)]

In [134]:
{2,3,4,5,6}-{2,1,6}

{3, 4, 5}

In [92]:
def suggest_rand(pairs, bus_list, n=3):
    out = []
    for i in range(n):
        a = random.choice(bus_list)
        b = random.choice(bus_list[:a]+bus)
        while (a,b) in pairs+out or (b,a) in pairs+out or a==b:
            a = random.choice(bus_list)
            b = random.choice(bus_list)
        out.append((a,b))
    return out

This will loop as follows:
* generate a random network of variable node count, demands, generators, and generation capacity.
* run the Pandapower DC OPF solver
* run our pulp solver
* compare results
* if infeasible, each reports separately. Ideally they run or fall together

In [ ]:
def completeGraph(pairs, buses):
    node0 = buses[0]
    partners = get_partners(node0)

In [208]:
for trial in range(50):
    print(f"\n## # Running Simulated Grid Number {trial+1} # ##\n"*2)
    buses = np.arange(np.random.randint(6,12))
    n = len(buses)
    demands = {bus:np.random.randint(3,8)*20 for bus in buses}
    line_capacity = 1000
    line_expansion_cost = 1000
    obj = 'cost'
    cut = False
    ac_lines = new_pairs([], buses,
                            n = np.random.randint(len(buses)+3, 2*len(buses)+4))
    poss_ac_lines = new_pairs(ac_lines, buses, n = 1)
    poss_dc_lines = new_pairs([], buses, n = 1)
    dc_lines = []
    susceptances = {line : np.random.randint(10,20)*100 for line in ac_lines + poss_ac_lines}  
    
    gen_buses = np.random.choice(buses, np.random.randint(1, 3), replace = False)
    gen_capacity = {g : np.random.randint(5,9)*250* 
                           int(g in gen_buses) for g in buses}
    generation_cost = {g : np.random.randint(1,100) for g in buses if g in gen_buses}
    printout = True
    max_new_ac = 0 
    max_new_dc = 0
    label=datelabel
    save = False
    print('Total Demand:',np.sum(list(demands.values())))
    print('Gen Costs:', generation_cost)
    
    try:
        net = panda_DCOPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
                     susceptances = susceptances, 
                     line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
                     demands = demands, ac_lines = ac_lines, 
                     dc_lines = dc_lines,
                     generation_cost = generation_cost, printout = False)
        print('-->Panda Converged')
        print('Panda generators:')
        print({net.gen['bus'][i]: net.res_gen['p_mw'][i] for i in net.gen.index })
    except Exception:
        print('Panda Failed')
    out, prob = flow.ACDC_TEP_OPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
             susceptances = susceptances, 
             line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
             demands = demands, ac_lines = ac_lines, 
             poss_ac_lines = poss_ac_lines, dc_lines = dc_lines, poss_dc_lines = poss_dc_lines, 
             generation_cost = generation_cost, printout = False,
             max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False)
    if prob.status !=1:
        print('-->Our Model Failed')
        keystr = 'gen_capacity line_capacity susceptances line_expansion_cost buses demands ac_lines poss_ac_lines poss_dc_lines generation_cost'
        keys = keystr.split(' ')
        values = [gen_capacity, line_capacity, susceptances, line_expansion_cost,
                  buses, demands, ac_lines, poss_ac_lines, poss_dc_lines, generation_cost]
        totD, totG= np.sum(list(demands.values())), np.sum(list(gen_capacity.values()))
        file = open(f'DidNotConverge/totD{totD}-totG{totG}-{datelabel}.txt','w')
        for i in range(len(keys)):
            file.write(f'{keys[i]}={values[i]}\n')
        file.close()
    else:
        print('-->Our Model Converged')
        flow_comp = pd.DataFrame({'PandaPower MW':list(net.res_line['p_from_mw'].values), 
                              'OurModel MW':list(out['ac_flow'].values())})
        flow_comp['Error MW'] = flow_comp['PandaPower MW']-flow_comp['OurModel MW']
        phase_comp = pd.DataFrame({'PandaPower rads':list(net.res_bus['va_degree'].values*np.pi/180), 
                                  'OurModel rads':list(out['phase'].values())})
        phase_comp['Error rads'] = phase_comp['PandaPower rads']-phase_comp['OurModel rads']
        sum_abs_flow_err = np.sum(np.abs(flow_comp['Error MW'].values))
        sum_abs_phase_err = np.sum(np.abs(phase_comp['Error rads'].values))
        print("Our generators:")
        print({key:out['gen'][key] for key in out['gen'] if gen_capacity[key] > 0})
        # print('Panda generators:')
        # print({net.gen['bus'][i]: net.res_gen['p_mw'][i] for i in net.gen.index })
        print("Line Flows' Absolute Error :\n", sum_abs_flow_err)
        print("Bus Phases:\n", sum_abs_phase_err)
    



## # Running Simulated Grid Number 1 # ##

## # Running Simulated Grid Number 1 # ##

Total Demand: 940
Gen Costs: {7: 76}
-->Panda Converged
Panda generators:
{7: 940.0}
-->Our Model Converged
Our generators:
{7: 939.9999999999999}
Line Flows' Absolute Error :
 6.394884621840902e-13
Bus Phases:
 2.636779683484747e-16

## # Running Simulated Grid Number 2 # ##

## # Running Simulated Grid Number 2 # ##

Total Demand: 1120
Gen Costs: {5: 76, 9: 52}
-->Panda Converged
Panda generators:
{5: 8.324886517281551e-06, 9: 1119.9999916751135}
-->Our Model Converged
Our generators:
{5: 0.0, 9: 1120.0}
Line Flows' Absolute Error :
 1.95256437720559e-05
Bus Phases:
 6.803304485741846e-08

## # Running Simulated Grid Number 3 # ##

## # Running Simulated Grid Number 3 # ##

Total Demand: 500
Gen Costs: {3: 46, 4: 54}
-->Panda Converged
Panda generators:
{3: 499.99999654406315, 4: 3.455937017577807e-06}
-->Our Model Converged
Our generators:
{3: 500.0, 4: 0.0}
Line Flows' Absolute Error :
 5.9744803

In [254]:
ValResults = {'panda':[], 'our':[], 'nodes': [], 'lines':[], 
              'total demand':[], 'total capacity':[], 'gen error': [],
             'phase error': [], 'objective error': [], 'flow error': []}
for trial in range(200):
    # print(f"\n## # Running Simulated Grid Number {trial+1} # ##\n"*2)
    buses = np.arange(np.random.randint(6,12))
    n = len(buses)
    demands = {bus:np.random.randint(3,8)*20 for bus in buses}
    line_capacity = 1000
    line_expansion_cost = 1000
    obj = 'cost'
    cut = False
    ac_lines = new_pairs([], buses,
                            n = np.random.randint(len(buses)+3, 2*len(buses)+4))
    poss_ac_lines = new_pairs(ac_lines, buses, n = 1)
    poss_dc_lines = new_pairs([], buses, n = 1)
    dc_lines = []
    susceptances = {line : np.random.randint(10,20)*100 for line in ac_lines + poss_ac_lines}  
    
    gen_buses = np.random.choice(buses, np.random.randint(1, 5), replace = False)
    gen_capacity = {g : np.random.randint(5,9)*150* 
                           int(g in gen_buses) for g in buses}
    generation_cost = {g : np.random.randint(1,100) for g in buses if g in gen_buses}
    printout = True
    max_new_ac = 0 
    max_new_dc = 0
    label=datelabel
    save = False
    # print('Total Demand:',np.sum(list(demands.values())))
    # print('Gen Costs:', generation_cost)
    ValResults['nodes'].append( n)
    ValResults['lines'].append(len(ac_lines))
    ValResults['total demand'].append(np.sum(list(demands.values())))
    ValResults['total capacity'].append( np.sum(list(gen_capacity.values())))
    
    try:
        net = panda_DCOPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
                     susceptances = susceptances, 
                     line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
                     demands = demands, ac_lines = ac_lines, 
                     dc_lines = dc_lines,
                     generation_cost = generation_cost, printout = False)
        # print('-->Panda Converged')
        ValResults['panda'].append(1)
        # print('Panda generators:')
        # print({net.gen['bus'][i]: net.res_gen['p_mw'][i] for i in net.gen.index })
        panda_gen = {net.gen['bus'][i]: net.res_gen['p_mw'][i] for i in net.gen.index }
    except Exception:
        # print('Panda Failed')
        ValResults['panda'].append(0)
        
    out, prob = flow.ACDC_TEP_OPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
             susceptances = susceptances, 
             line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
             demands = demands, ac_lines = ac_lines, 
             poss_ac_lines = poss_ac_lines, dc_lines = dc_lines, poss_dc_lines = poss_dc_lines, 
             generation_cost = generation_cost, printout = False,
             max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False)
    if prob.status !=1:
        # print('-->Our Model Failed')
        ValResults['our'].append(0)
        keystr = 'gen_capacity line_capacity susceptances line_expansion_cost buses demands ac_lines poss_ac_lines poss_dc_lines generation_cost'
        keys = keystr.split(' ')
        values = [gen_capacity, line_capacity, susceptances, line_expansion_cost,
                  buses, demands, ac_lines, poss_ac_lines, poss_dc_lines, generation_cost]
        totD, totG= np.sum(list(demands.values())), np.sum(list(gen_capacity.values()))
        file = open(f'DidNotConverge/totD{totD}-totG{totG}-{datelabel}.txt','w')
        for i in range(len(keys)):
            file.write(f'{keys[i]}={values[i]}\n')
        file.close()
        
        ValResults['gen error'].append(-1)
        ValResults['phase error'].append(-1)
        ValResults['flow error'].append(-1)
        ValResults['objective error'].append(-1)
    else:
        # print('-->Our Model Converged')
        ValResults['our'].append(1)
        flow_comp = pd.DataFrame({'PandaPower MW':list(net.res_line['p_from_mw'].values), 
                              'OurModel MW':list(out['ac_flow'].values())})
        flow_comp['Error MW'] = flow_comp['PandaPower MW']-flow_comp['OurModel MW']
        phase_comp = pd.DataFrame({'PandaPower rads':list(net.res_bus['va_degree'].values*np.pi/180), 
                                  'OurModel rads':list(out['phase'].values())})
        phase_comp['Error rads'] = phase_comp['PandaPower rads']-phase_comp['OurModel rads']
        sum_abs_flow_err = np.sum(np.abs(flow_comp['Error MW'].values))
        sum_abs_phase_err = np.sum(np.abs(phase_comp['Error rads'].values))
        # print("Our generators:")
        # print({key:out['gen'][key] for key in out['gen'] if gen_capacity[key] > 0})
        our_gen = {key:out['gen'][key] for key in out['gen'] if gen_capacity[key] > 0}
        sum_gen_err = np.sum([panda_gen[g] for g in our_gen]) - np.sum([our_gen[g] for g in our_gen]) 
        
        # print('Panda generators:')
        # print({net.gen['bus'][i]: net.res_gen['p_mw'][i] for i in net.gen.index })
        # print("Line Flows' Absolute Error :\n", sum_abs_flow_err)
        # print("Bus Phases:\n", sum_abs_phase_err)
        ValResults['phase error'].append( sum_abs_phase_err)
        ValResults['flow error'].append(sum_abs_flow_err)
        ValResults['gen error'].append(sum_gen_err)
        ValResults['objective error'].append(net.res_cost-prob.objective.value())
                                      
        

In [258]:
pd.DataFrame(ValResults).to_csv('Validate18april2025.csv')

In [222]:
prob.objective.value()

25200.0

In [226]:
net.res_cost

25200.00001785033

In [94]:
vis=[]
node0 = ac_lines[0][0]
for pair in ac_lines:
    if node0 == pair[0]:
        vis.append(pair[1])
    elif node0== pair[1]:
        vis.append(pair[0])

In [96]:
vis

[1, 5, 3, 2]

In [55]:
{net.gen['bus'][i]: net.res_gen['p_mw'][i] for i in net.gen.index }

{1: 419.99999958398973, 3: 4.160102549538972e-07}

In [51]:
net.gen.index

Index([0, 1], dtype='int64')

In [102]:
net.res_load

,p_mw,q_mvar
0,140.0,NaN
1,60.0,NaN
2,80.0,NaN
3,80.0,NaN
4,80.0,NaN
5,80.0,NaN
6,100.0,NaN
7,0.0,NaN
8,80.0,NaN


In [108]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,32.841452,0.0,-32.841452,0.0,0.0,0.0,18.961021,18.961021,18.961021,1.0,-7.829529,1.0,-9.083980,1.896102
1,-51.139810,0.0,51.139810,0.0,0.0,0.0,29.525583,29.525583,29.525583,1.0,-10.493252,1.0,-7.829529,2.952558
2,66.317569,0.0,-66.317569,0.0,0.0,0.0,38.288467,38.288467,38.288467,1.0,-6.986149,1.0,-9.221277,3.828847
3,0.000000,0.0,-0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,1.0,-10.493252,1.0,-10.493252,0.000000
4,-146.317569,0.0,146.317569,0.0,0.0,0.0,84.476488,84.476488,84.476488,1.0,-6.986149,1.0,0.000000,8.447649
5,28.860190,0.0,-28.860190,0.0,0.0,0.0,16.662438,16.662438,16.662438,1.0,-9.221277,1.0,-10.493252,1.666244
6,0.000000,0.0,-0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,1.0,-6.832325,1.0,-6.832325,0.000000
7,0.000000,0.0,0.000000,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,62.542620,0.0,-62.542620,0.0,0.0,0.0,36.108999,36.108999,36.108999,1.0,-6.832325,1.0,-9.221277,3.610900
9,-47.158548,0.0,47.158548,0.0,0.0,0.0,27.227001,27.227001,27.227001,1.0,-9.083980,1.0,-6.832325,2.722700


In [ ]:
net

In [70]:
out, prob = flow.ACDC_TEP_OPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
         susceptances = susceptances, 
         line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
         demands = demands, ac_lines = ac_lines, 
         poss_ac_lines = poss_ac_lines, dc_lines = dc_lines, poss_dc_lines = poss_dc_lines, 
         generation_cost = generation_cost, printout = False,
         max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False)

In [76]:
prob.status


0

In [84]:
net.gen

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,min_p_mw,max_p_mw
0,Gen Bus 1,1,0.0,1.02,NaN,-10.0,10.0,1.0,True,True,0.0,None,True,0.0,2000.0


In [37]:
buses = np.arange(np.random.randint(5,12))
n = len(buses)
demands = {bus:np.random.randint(3,8)*20 for bus in buses}
line_capacity = 1000
line_expansion_cost = 1000
obj = 'cost'
cut = False
ac_lines = suggest_rand([], buses,
                        n = np.random.randint(len(buses)+4, 2*len(buses)+4))
poss_ac_lines = suggest_rand(ac_lines, buses, n = 1)
poss_dc_lines = suggest_rand([], buses, n = 1)
dc_lines = []
susceptances = {line : np.random.randint(10,20)*100 for line in ac_lines + poss_ac_lines}  

gen_buses = np.random.choice(buses, np.random.randint(1, 3), replace = False)
gen_capacity = {g : np.random.randint(5,9)*250* 
                       int(g in gen_buses) for g in buses}
generation_cost = {g : np.random.randint(1,100) for g in buses if g in gen_buses}
printout = True
max_new_ac = 0 
max_new_dc = 0
label=datelabel
save = False
print('Total Demand:',np.sum(list(demands.values())))
print('Gen Costs:', generation_cost)

net = panda_DCOPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
                 susceptances = susceptances, 
                 line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
                 demands = demands, ac_lines = ac_lines, 
                 dc_lines = dc_lines,
                 generation_cost = generation_cost, printout = False)

Total Demand: 520
Gen Costs: {1: 22, 4: 89}


In [112]:
out, net = panda_DCOPF(gen_capacity = gen_capacity, line_capacity = line_capacity, 
                 susceptances = susceptances, 
                 line_expansion_cost = line_expansion_cost, buses = buses, obj = 'cost', cut = False,
                 demands = demands, ac_lines = ac_lines, 
                 dc_lines = dc_lines,
                 generation_cost = generation_cost, printout = False)

In [114]:
net.res_gen

AttributeError: 'NoneType' object has no attribute 'res_gen'

In [49]:
 {'PandaPower MW':list(net.res_line['p_from_mw'].values), 
           'OurModel MW':list(out['ac_flow'].values())}

{'PandaPower MW': [59.9999913714589, 39.999994244336634, 19.999997121743547],
 'OurModel MW': [12.985536551073551,
  178.80886970514325,
  -18.18853429251078,
  -81.66747037414291,
  0.0,
  78.86330103734281,
  51.215261642261794,
  -43.96316298129386,
  15.08577663900337,
  -9.98730236773099,
  25.34699307478354,
  -284.13929496500293,
  58.653974724380305,
  8.641277092111409,
  -27.593608062881458,
  -237.05183532985356,
  -58.04322117769698,
  0.0,
  0.0]}

In [168]:
np.sum(list(gen_capacity.values()))

2500

In [172]:
(net.res_gen['p_mw']**2 + net.res_gen['p_mw']**2)**.5

0    1781.908379
1       0.000710
Name: p_mw, dtype: float64

In [202]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,-116.265017,-8.421379,116.265017,19.150610,-1.421085e-14,10.729231,68.186020,68.186020,68.186020,0.987028,-21.258050,0.997714,-16.047401,6.818602
1,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,66.808625,12.530220,-66.808625,-10.078004,0.000000e+00,2.452215,39.409005,39.409005,39.409005,0.995826,-25.650528,0.989834,-27.694843,3.940900
3,61.674873,3.145255,-61.674873,-0.333409,0.000000e+00,2.811846,36.224229,36.224229,36.224229,0.984266,-26.222680,0.983003,-28.832351,3.622423
4,39.674723,-1.956759,-39.674723,2.806560,0.000000e+00,0.849801,23.199298,23.199298,23.199298,0.988567,-26.472065,0.989834,-27.694843,2.319930
5,-43.516652,7.271444,43.516652,-5.615809,-7.105427e-15,1.655636,25.734301,25.734301,25.734301,0.989834,-27.694843,0.984394,-25.561966,2.573430
6,142.926092,10.830179,-142.926092,1.574967,0.000000e+00,12.405146,83.842606,83.842606,83.842606,0.987028,-21.258050,0.984266,-26.222680,8.384261
7,-414.846383,-48.630698,414.846383,173.818473,0.000000e+00,125.187775,241.704285,241.704285,241.704285,0.997714,-16.047401,1.074395,0.000000,24.170429
8,13.407395,0.228644,-13.407395,-0.074013,0.000000e+00,0.154631,7.864623,7.864623,7.864623,0.984394,-25.561966,0.984266,-26.222680,0.786462
9,-4.658614,4.686831,4.658614,-4.646209,0.000000e+00,0.040623,3.859405,3.859405,3.859405,0.988567,-26.472065,0.984266,-26.222680,0.385941


In [204]:
out['ac_flow']

{(3, 4): 85.35791285973482,
 (1, 5): 302.79127436866753,
 (2, 0): -36.80720624275659,
 (6, 6): 0.0,
 (0, 1): -254.46854468067968,
 (7, 0): 38.42899967259527,
 (2, 4): 89.02931766249776,
 (3, 5): -87.17850489089983,
 (1, 2): 274.3111812780575,
 (6, 5): -125.61276947776764,
 (0, 3): 76.09033811051859,
 (2, 3): 42.08906985831641,
 (7, 1): -158.42899967259527,
 (6, 4): -24.387230522232358,
 (5, 4): 0.0}

In [156]:
line_capacity = 1000
line_expansion_cost = 1000
obj = 'cost'
cut = False
(ac_lines ,poss_ac_lines,poss_dc_lines ,susceptances  ,gen_buses ,gen_capacity ,
generation_cost, demands)

([(10, 7),
  (5, 5),
  (0, 4),
  (3, 9),
  (1, 4),
  (4, 2),
  (10, 3),
  (7, 6),
  (2, 3),
  (1, 3),
  (9, 2),
  (0, 8),
  (7, 2),
  (10, 1),
  (6, 8),
  (10, 6),
  (1, 0),
  (10, 2)],
 [(6, 3)],
 [(3, 6)],
 {(10, 7): 1300,
  (5, 5): 1300,
  (0, 4): 1900,
  (3, 9): 1400,
  (1, 4): 1900,
  (4, 2): 1200,
  (10, 3): 1700,
  (7, 6): 1400,
  (2, 3): 1200,
  (1, 3): 1100,
  (9, 2): 1600,
  (0, 8): 1200,
  (7, 2): 1100,
  (10, 1): 1800,
  (6, 8): 1200,
  (10, 6): 1100,
  (1, 0): 1800,
  (10, 2): 1600,
  (6, 3): 1200},
 array([8, 6]),
 {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 1500, 7: 0, 8: 1000, 9: 0, 10: 0},
 {6: 56, 8: 59},
 {0: 150,
  1: 150,
  2: 150,
  3: 90,
  4: 150,
  5: 90,
  6: 90,
  7: 120,
  8: 90,
  9: 150,
  10: 120})

In [86]:
np.sum(list(demands.values()))

810

In [118]:
buses = np.arange(np.random.randint(4,10))
n = len(buses)
demands = {bus:np.random.randint(3,7)*30 for bus in buses}
np.sum(list(demands.values())), np.sum(list(gen_capacity.values()))


(990, 750)

In [120]:
datelabel

'25-04-16'

In [122]:
pulp.LpStatus

{0: 'Not Solved',
 1: 'Optimal',
 -1: 'Infeasible',
 -2: 'Unbounded',
 -3: 'Undefined'}